# Entertainment Industry Performance Analytics - Data Exploration and Cleaning

**Objective:** Combine two real, independently sourced datasets to support both a studio/genre
performance analysis and an individual-viewer engagement lifetime value model.

**Data sources:**
1. TMDB 5000 Movie Dataset - real budget, revenue, genre, and production company data for 4,803 films.
2. MovieLens `ml-latest-small` - 100,836 real, timestamped ratings from 610 real users on 9,742 movies (1996-2018), from GroupLens.

The two datasets are joined through `links.csv`, which maps MovieLens `movieId` to the same TMDB `id
used in the financial dataset, giving genuine per-viewer engagement history for movies with real box
office outcomes.

**Outputs:** `tmdb_clean.csv`, `ratings_clean.csv`, `ratings_financial.csv`.

## 1. Setup

In [1]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt

pd.set_option('display.width', 140)
plt.style.use('seaborn-v0_8-whitegrid')
print("Setup complete.")

Setup complete.


## 2. Load and Inspect TMDB Financial Data

In [2]:
tmdb = pd.read_csv("../data/tmdb_5000_movies.csv")
print("Raw shape:", tmdb.shape)
tmdb[['title','budget','revenue','release_date','vote_average','vote_count','popularity']].head()

Raw shape: (4803, 20)


,title,budget,revenue,release_date,vote_average,vote_count,popularity
0,Avatar,237000000,2787965087,2009-12-10,7.2,11800,150.437577
1,Pirates of the Caribbean: At World's End,300000000,961000000,2007-05-19,6.9,4500,139.082615
2,Spectre,245000000,880674609,2015-10-26,6.3,4466,107.376788
3,The Dark Knight Rises,250000000,1084939099,2012-07-16,7.6,9106,112.312950
4,John Carter,260000000,284139100,2012-03-07,6.1,2124,43.926995


## 3. Data Quality Issues

The TMDB dataset uses `0` as a placeholder for unknown budget or revenue rather than a null value,
which would understate financial coverage if left in. Genres and production companies are stored as
JSON strings rather than plain columns.

In [3]:
print("Rows with budget == 0:", (tmdb['budget'] == 0).sum())
print("Rows with revenue == 0:", (tmdb['revenue'] == 0).sum())
print("Rows with both budget and revenue populated (>0):", ((tmdb['budget'] > 0) & (tmdb['revenue'] > 0)).sum())
print("Missing release_date:", tmdb['release_date'].isna().sum())

Rows with budget == 0: 1037
Rows with revenue == 0: 1427
Rows with both budget and revenue populated (>0): 3229
Missing release_date: 1


## 4. Clean the Financial Data

In [4]:
tmdb_clean = tmdb[(tmdb['budget'] > 10000) & (tmdb['revenue'] > 0)].copy()
tmdb_clean['release_date'] = pd.to_datetime(tmdb_clean['release_date'], errors='coerce')
tmdb_clean = tmdb_clean.dropna(subset=['release_date'])
tmdb_clean['release_year'] = tmdb_clean['release_date'].dt.year
tmdb_clean['profit'] = tmdb_clean['revenue'] - tmdb_clean['budget']
tmdb_clean['roi'] = tmdb_clean['profit'] / tmdb_clean['budget']
tmdb_clean['profitable'] = tmdb_clean['profit'] > 0

def parse_json_names(s):
    try:
        return [x['name'] for x in json.loads(s)]
    except Exception:
        return []

tmdb_clean['genre_list'] = tmdb_clean['genres'].apply(parse_json_names)
tmdb_clean['studio_list'] = tmdb_clean['production_companies'].apply(parse_json_names)
tmdb_clean['primary_studio'] = tmdb_clean['studio_list'].apply(lambda x: x[0] if x else None)

print("Clean shape:", tmdb_clean.shape)
print("Year range:", tmdb_clean['release_year'].min(), "-", tmdb_clean['release_year'].max())
print("Share of financially-tracked films that were profitable:", round(tmdb_clean['profitable'].mean(), 4))

Clean shape: (3213, 27)
Year range: 1916 - 2016
Share of financially-tracked films that were profitable: 0.7557


## 5. Load and Inspect MovieLens Rating Data

In [5]:
ratings = pd.read_csv("../data/ratings.csv")
links = pd.read_csv("../data/links.csv")
movies = pd.read_csv("../data/movies.csv")

ratings = ratings.merge(links[['movieId', 'tmdbId']], on='movieId', how='left')
ratings['tmdbId'] = ratings['tmdbId'].astype('Int64')
ratings['rated_at'] = pd.to_datetime(ratings['timestamp'], unit='s')

print("Ratings shape:", ratings.shape)
print("Distinct users:", ratings['userId'].nunique())
print("Distinct movies rated:", ratings['movieId'].nunique())
print("Date range:", ratings['rated_at'].min(), "to", ratings['rated_at'].max())
ratings.head()

Ratings shape: (100836, 6)
Distinct users: 610
Distinct movies rated: 9724
Date range: 1996-03-29 18:36:55 to 2018-09-24 14:27:30


,userId,movieId,rating,timestamp,tmdbId,rated_at
0,1,1,4.0,964982703,862,2000-07-30 18:45:03
1,1,3,4.0,964981247,15602,2000-07-30 18:20:47
2,1,6,4.0,964982224,949,2000-07-30 18:37:04
3,1,47,5.0,964983815,807,2000-07-30 19:03:35
4,1,50,5.0,964982931,629,2000-07-30 18:48:51


## 6. Join Ratings to Real Financial Data

Every rating is matched to its TMDB `id` through `links.csv`. Only ratings for movies that also appear
in the cleaned financial dataset are kept for the revenue-linked engagement analysis; the full ratings
table is retained separately for the engagement lifetime value model, which does not require financial
data.

In [6]:
tmdb_ids = set(tmdb_clean['id'].astype(int))
ratings_financial = ratings[ratings['tmdbId'].isin(tmdb_ids)].copy()

print("Ratings with matching real financial data:", ratings_financial.shape)
print("Distinct financially-tracked movies rated:", ratings_financial['movieId'].nunique())
print("Distinct users with at least one financially-tracked rating:", ratings_financial['userId'].nunique(), "of", ratings['userId'].nunique(), "total users")

Ratings with matching real financial data: (65355, 6)
Distinct financially-tracked movies rated: 2803
Distinct users with at least one financially-tracked rating: 610 of 610 total users


## 7. Exploratory Views

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
tmdb_clean.groupby('release_year').size().plot(ax=axes[0], title='Financially-Tracked Films per Year')
axes[0].set_ylabel('Films')
ratings.set_index('rated_at').resample('YE').size().plot(ax=axes[1], title='MovieLens Ratings per Year')
axes[1].set_ylabel('Ratings')
plt.tight_layout()
plt.savefig('eda_overview.png', dpi=100)
plt.show()

<Figure size 1200x450 with 2 Axes>

## 8. Save Cleaned Data

In [8]:
tmdb_clean.to_csv("../data/tmdb_clean.csv", index=False)
ratings.to_csv("../data/ratings_clean.csv", index=False)
ratings_financial.to_csv("../data/ratings_financial.csv", index=False)
print("Saved tmdb_clean.csv, ratings_clean.csv, ratings_financial.csv")

Saved tmdb_clean.csv, ratings_clean.csv, ratings_financial.csv
